# EU Critical Raw Materials: Import Dependency and Supply Chain Risk Analysis (2019 - 2025)

## 1. Project Overview

### Business Problem
The European Union depends on international suppliers for critical raw materials used in manufacturing, batteries, electronics and other industries. Heavy dependence on a small number of supplier countries can increase supply-chain risk.

### Project Objective
Analyse EU import data from 2019 to 2025 to understand supplier dependency and identify materials with high supplier concentration.

### Materials Analysed
- Magnesium
- Gallium
- Natural graphite
- Lithium carbonate
- Copper

### Key Questions
1. Which countries supply the largest share of EU imports?
2. How has supplier concentration changed between 2019 and 2025?
3. Which materials should be prioritised for supplier diversification?

### Data and Method
The project uses Eurostat Comext trade data, measured in euros. Python and pandas will be used to inspect and prepare the data, calculate supplier shares, and measure concentration using the Herfindahl–Hirschman Index (HHI).

### Expected Outcome
A clear, data-driven comparison of supplier concentration across five materials, supported by charts, findings and practical supply-chain recommendations.

In [ ]:
import pandas as pd


In [ ]:
import os

print(os.listdir())

['.config', 'ds-045409__custom_22978287_spreadsheet.xlsx', 'ds-045409__custom_22978350_spreadsheet.xlsx', 'ds-045409__custom_22978315_spreadsheet.xlsx', 'ds-045409__custom_22978295_spreadsheet.xlsx', 'ds-045409__custom_22978252_spreadsheet.xlsx', 'CRM_update_June_2026_v2.xlsx', 'sample_data']


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CRM_update_June_2026_v2.xlsx to CRM_update_June_2026_v2.xlsx
Saving ds-045409__custom_22978252_spreadsheet.xlsx to ds-045409__custom_22978252_spreadsheet.xlsx
Saving ds-045409__custom_22978287_spreadsheet.xlsx to ds-045409__custom_22978287_spreadsheet.xlsx
Saving ds-045409__custom_22978295_spreadsheet.xlsx to ds-045409__custom_22978295_spreadsheet.xlsx
Saving ds-045409__custom_22978315_spreadsheet.xlsx to ds-045409__custom_22978315_spreadsheet.xlsx
Saving ds-045409__custom_22978350_spreadsheet.xlsx to ds-045409__custom_22978350_spreadsheet.xlsx


In [ ]:
files = {
    "Magnesium": "ds-045409__custom_22978252_spreadsheet.xlsx",
    "Gallium": "ds-045409__custom_22978287_spreadsheet.xlsx",
    "Natural graphite": "ds-045409__custom_22978295_spreadsheet.xlsx",
    "Lithium carbonate": "ds-045409__custom_22978315_spreadsheet.xlsx",
    "Copper": "ds-045409__custom_22978350_spreadsheet.xlsx"
}

for material, filename in files.items():
    sheets = pd.ExcelFile(filename).sheet_names
    print(material, ":", sheets)

Magnesium : ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7']
Gallium : ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7']
Natural graphite : ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7']
Lithium carbonate : ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7']
Copper : ['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7', 'Sheet 8', 'Sheet 9', 'Sheet 10', 'Sheet 11', 'Sheet 12', 'Sheet 13', 'Sheet 14']


In [ ]:
magnesium = pd.read_excel(
    files["Magnesium"],
    sheet_name="Sheet 1",
    header=None
)

print(magnesium.iloc[8:16, :5].to_string(index=False))

                   0                                                               1                                                               2                                                              3                                                              4
                 NaN                                                             NaN                                                             NaN                                                            NaN                                                            NaN
     PRODUCT (Codes)                                                        81041100                                                        81041100                                                       81041900                                                       81041900
    PRODUCT (Labels) Unwrought magnesium, containing >= 99,8% by weight of magnesium Unwrought magnesium, containing >= 99,8% by weight of magnesium Unwrought magnesium, conta

In [ ]:
magnesium_imports = magnesium.iloc[13:, [0, 1, 3]].copy()

magnesium_imports.columns = [
    "partner", "product_81041100", "product_81041900"
]

print(magnesium_imports.head())

                 partner product_81041100 product_81041900
13               Andorra                :                :
14  United Arab Emirates               14                :
15           Afghanistan                :                :
16   Antigua and Barbuda                :                :
17              Anguilla                :                :


In [ ]:
magnesium_imports["product_81041100"] = pd.to_numeric(
    magnesium_imports["product_81041100"].replace(":", pd.NA),
    errors="coerce"
)

magnesium_imports["product_81041900"] = pd.to_numeric(
    magnesium_imports["product_81041900"].replace(":", pd.NA),
    errors="coerce"
)

print(magnesium_imports.dtypes)

partner              object
product_81041100    float64
product_81041900    float64
dtype: object


In [ ]:
print(magnesium_imports.isna().sum())

partner               1
product_81041100    233
product_81041900    232
dtype: int64


### Step 1 - Remove Empty Records

**Objective:** Remove records that cannot contribute to the supplier analysis.

**Action:** Remove rows without a supplier name and rows where both magnesium import values are missing.

**Why it matters:** We need identifiable suppliers and recorded import values for our analysis.

In [ ]:
magnesium_imports = magnesium_imports.dropna(
    subset=["partner", "product_81041100", "product_81041900"],
    how="all"
)

print("Remaining rows:", len(magnesium_imports))

Remaining rows: 245


### Step 2 - Add Year Information

**Objective:** Identify the year associated with each supplier record.

**Action:** Add a `year` column with the value 2019.

**Why it matters:** We will eventually combine seven years of data and analyse changes over time.

In [ ]:
magnesium_imports["year"] = 2019

print(magnesium_imports.head())

                 partner  product_81041100  product_81041900  year
13               Andorra               NaN               NaN  2019
14  United Arab Emirates              14.0               NaN  2019
15           Afghanistan               NaN               NaN  2019
16   Antigua and Barbuda               NaN               NaN  2019
17              Anguilla               NaN               NaN  2019


### Step 3 - Review the Prepared Magnesium Data

**Objective:** Verify that the cleaned 2019 magnesium data has the expected structure.

**Action:** Display the number of rows, columns and the first few records.

**Why it matters:** This provides a final check before preparing the remaining six years.

In [ ]:
print("Rows and columns:", magnesium_imports.shape)
print(magnesium_imports.head())

Rows and columns: (245, 4)
                 partner  product_81041100  product_81041900  year
13               Andorra               NaN               NaN  2019
14  United Arab Emirates              14.0               NaN  2019
15           Afghanistan               NaN               NaN  2019
16   Antigua and Barbuda               NaN               NaN  2019
17              Anguilla               NaN               NaN  2019


### Step 4 - Combine Magnesium Data (2019–2025)

**Objective:** Prepare one dataset containing seven years of magnesium imports.

**Action:** Read each annual Excel sheet, extract supplier countries and import values, convert values into numeric format, and add the corresponding year.

**Why it matters:** A combined dataset allows us to analyse changes in supplier dependency over time.

In [ ]:
magnesium_years = []

for year in range(2019, 2026):
    sheet = f"Sheet {year - 2018}"

    df = pd.read_excel(
        files["Magnesium"],
        sheet_name=sheet,
        header=None
    )

    data = df.iloc[13:, [0, 1, 3]].copy()
    data.columns = ["partner", "product_81041100", "product_81041900"]

    data["product_81041100"] = pd.to_numeric(
        data["product_81041100"].replace(":", pd.NA),
        errors="coerce"
    )

    data["product_81041900"] = pd.to_numeric(
        data["product_81041900"].replace(":", pd.NA),
        errors="coerce"
    )

    data["year"] = year

    data = data.dropna(
        subset=["partner", "product_81041100", "product_81041900"],
        how="all"
    )

    magnesium_years.append(data)

magnesium_all = pd.concat(magnesium_years, ignore_index=True)

print(magnesium_all.shape)
print(sorted(magnesium_all["year"].unique()))

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(1715, 4)
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Step 5 - Validate the Combined Magnesium Dataset

**Objective:** Check whether the combined dataset contains all seven years and identify missing values.

**Action:** Review the dataset's dimensions, year coverage and missing values.

**Why it matters:** We need to check data completeness before calculating supplier dependency.

In [ ]:
print(magnesium_all.isna().sum())

partner                0
product_81041100    1616
product_81041900    1614
year                   0
dtype: int64


### Step 6 - Check Available Import Records

**Objective:** Identify how many actual import values are available for each magnesium product across 2019–2025.

**Action:** Count the non-missing import values for both magnesium product categories.

**Why it matters:** This helps us understand how much usable data is available before calculating supplier dependency.

In [ ]:
print("Product 81041100:", magnesium_all["product_81041100"].count())
print("Product 81041900:", magnesium_all["product_81041900"].count())

Product 81041100: 99
Product 81041900: 101


### Step 7 - Prepare Gallium Import Data

**Objective:** Extract gallium import data from 2019 to 2025.

**Action:** Read the seven annual worksheets, extract supplier countries and import values, convert values into numeric format, and add the corresponding year.

**Why it matters:** We need the same supplier-level data structure for all five materials to compare their import dependencies.

In [ ]:
gallium_years = []

for year in range(2019, 2026):
    sheet = f"Sheet {year - 2018}"

    df = pd.read_excel(
        files["Gallium"],
        sheet_name=sheet,
        header=None
    )

    data = df.iloc[12:, [0, 1]].copy()
    data.columns = ["partner", "import_value"]

    data["import_value"] = pd.to_numeric(
        data["import_value"].replace(":", pd.NA),
        errors="coerce"
    )

    data["year"] = year
    data = data.dropna(subset=["partner", "import_value"])

    gallium_years.append(data)

gallium_all = pd.concat(gallium_years, ignore_index=True)

print(gallium_all.shape)
print(sorted(gallium_all["year"].unique()))

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(74, 3)
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Step 8 - Prepare Natural Graphite Import Data

**Objective:** Extract natural graphite import data for 2019–2025.

**Action:** Combine the seven annual worksheets, retaining supplier countries and both graphite product categories (250410 and 250490).

**Why it matters:** Natural graphite has two product categories, so we must preserve both when preparing the dataset.

In [ ]:
graphite_years = []

for year in range(2019, 2026):
    sheet = f"Sheet {year - 2018}"

    df = pd.read_excel(
        files["Natural graphite"],
        sheet_name=sheet,
        header=None
    )

    data = df.iloc[13:, [0, 1, 3]].copy()
    data.columns = ["partner", "product_250410", "product_250490"]

    for col in ["product_250410", "product_250490"]:
        data[col] = pd.to_numeric(
            data[col].replace(":", pd.NA),
            errors="coerce"
        )

    data["year"] = year
    data = data.dropna(
        subset=["partner", "product_250410", "product_250490"],
        how="all"
    )

    graphite_years.append(data)

graphite_all = pd.concat(graphite_years, ignore_index=True)

print(graphite_all.shape)
print(sorted(graphite_all["year"].unique()))

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

(1715, 4)
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


### Step 9 - Prepare Lithium Carbonate Import Data

**Objective:** Extract lithium carbonate import data for 2019–2025.

**Action:** Combine the seven annual worksheets, retaining supplier countries and import values for product code 28369100.

**Why it matters:** This creates a consistent annual supplier dataset for lithium carbonate analysis.

In [ ]:
lithium_years = []

for year in range(2019, 2026):
    sheet = f"Sheet {year - 2018}"

    df = pd.read_excel(
        files["Lithium carbonate"],
        sheet_name=sheet,
        header=None
    )

    data = df.iloc[13:, [0, 1]].copy()
    data.columns = ["partner", "import_value"]

    data["import_value"] = pd.to_numeric(
        data["import_value"].replace(":", pd.NA),
        errors="coerce"
    )

    data["year"] = year
    data = data.dropna(subset=["partner", "import_value"])

    lithium_years.append(data)

lithium_all = pd.concat(lithium_years, ignore_index=True)

print(lithium_all.shape)
print(sorted(lithium_all["year"].unique()))

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

(126, 3)
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Step 10 - Prepare Copper Import Data

**Objective:** Extract copper import data for 2019–2025.

**Action:** Combine the seven annual import worksheets, retaining supplier countries and all five copper product categories.

**Why it matters:** Copper contains multiple product categories with different supplier distributions. Keeping them separate allows more accurate product-level analysis.

In [ ]:
copper_years = []

for year in range(2019, 2026):
    sheet = f"Sheet {year - 2018}"

    df = pd.read_excel(
        files["Copper"],
        sheet_name=sheet,
        header=None
    )

    data = df.iloc[13:, [0, 1, 2, 3, 4, 5]].copy()

    data.columns = [
        "partner", "product_26030000", "product_74010000",
        "product_74020000", "product_74031100", "product_74031300"
    ]

    for col in data.columns[1:]:
        data[col] = pd.to_numeric(
            data[col].replace(":", pd.NA),
            errors="coerce"
        )

    data["year"] = year
    data = data.dropna(subset=data.columns[1:6], how="all")

    copper_years.append(data)

copper_all = pd.concat(copper_years, ignore_index=True)

print(copper_all.shape)
print(sorted(copper_all["year"].unique()))

/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

(485, 7)
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Step 11 - Review Prepared Material Datasets

**Objective:** Review the dimensions of all five prepared datasets.

**Action:** Display the number of rows and columns for each material.

**Why it matters:** This provides an overview of the prepared data before combining it for supplier dependency analysis.

In [ ]:
print("Magnesium:", magnesium_all.shape)
print("Gallium:", gallium_all.shape)
print("Natural graphite:", graphite_all.shape)
print("Lithium carbonate:", lithium_all.shape)
print("Copper:", copper_all.shape)

Magnesium: (1715, 4)
Gallium: (74, 3)
Natural graphite: (1715, 4)
Lithium carbonate: (126, 3)
Copper: (485, 7)


In [ ]:
magnesium_all = magnesium_all.dropna(subset=["partner"])
magnesium_all = magnesium_all.dropna(
    subset=["product_81041100", "product_81041900"],
    how="all"
)

graphite_all = graphite_all.dropna(subset=["partner"])
graphite_all = graphite_all.dropna(
    subset=["product_250410", "product_250490"],
    how="all"
)

print("Magnesium:", magnesium_all.shape)
print("Natural graphite:", graphite_all.shape)

Magnesium: (132, 4)
Natural graphite: (247, 4)


In [ ]:
datasets = {
    "Magnesium": magnesium_all,
    "Gallium": gallium_all,
    "Natural graphite": graphite_all,
    "Lithium carbonate": lithium_all,
    "Copper": copper_all
}

for name, df in datasets.items():
    print("\n", name)
    print("Missing supplier names:", df["partner"].isna().sum())
    print("Missing years:", df["year"].isna().sum())
    print("Years available:", sorted(df["year"].unique()))


 Magnesium
Missing supplier names: 0
Missing years: 0
Years available: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

 Gallium
Missing supplier names: 0
Missing years: 0
Years available: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

 Natural graphite
Missing supplier names: 0
Missing years: 0
Years available: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

 Lithium carbonate
Missing supplier names: 0
Missing years: 0
Years available: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

 Copper
Missing supplier names: 0
Missing years: 0
Years available: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


### Step 12 - Check import-value data types

In [ ]:
value_columns = {
    "Magnesium": ["product_81041100", "product_81041900"],
    "Gallium": ["import_value"],
    "Natural graphite": ["product_250410", "product_250490"],
    "Lithium carbonate": ["import_value"],
    "Copper": [
        "product_26030000", "product_74010000",
        "product_74020000", "product_74031100",
        "product_74031300"
    ]
}

for name, df in datasets.items():
    print("\n", name)
    print(df[value_columns[name]].dtypes)


 Magnesium
product_81041100    float64
product_81041900    float64
dtype: object

 Gallium
import_value    float64
dtype: object

 Natural graphite
product_250410    float64
product_250490    float64
dtype: object

 Lithium carbonate
import_value    float64
dtype: object

 Copper
product_26030000    float64
product_74010000    float64
product_74020000    float64
product_74031100    float64
product_74031300    float64
dtype: object


###Step 13 -  Standardise Magnesium data

In [ ]:
magnesium_clean = magnesium_all.copy()

magnesium_clean["import_value"] = (
    magnesium_clean["product_81041100"].fillna(0) +
    magnesium_clean["product_81041900"].fillna(0)
)

magnesium_clean = magnesium_clean[
    ["partner", "year", "import_value"]
]

print(magnesium_clean.head())
print("Total import value:", magnesium_clean["import_value"].sum())

                                           partner  year  import_value
1                             United Arab Emirates  2019          14.0
17                                      Bangladesh  2019        1021.0
34                                          Canada  2019         847.0
39  Switzerland (incl. Liechtenstein 'LI' -> 1994)  2019        2670.0
44                                           China  2019   303119610.0
Total import value: 5778940637.0


###Step 14 -  Standardise Gallium data

In [ ]:
gallium_clean = gallium_all[
    ["partner", "year", "import_value"]
].copy()

print(gallium_clean.head())
print("Total import value:", gallium_clean["import_value"].sum())

                       partner  year  import_value
0                       Brazil  2019        1821.0
1                        China  2019     3733925.0
2       Extra-EU27 (from 2020)  2019     4499350.0
3               United Kingdom  2019       95956.0
4  Russian Federation (Russia)  2019        3556.0
Total import value: 124906994.0


###Step 15 -  Standardise Natural Graphite data

In [ ]:
graphite_clean = graphite_all.copy()

graphite_clean["import_value"] = (
    graphite_clean["product_250410"].fillna(0) +
    graphite_clean["product_250490"].fillna(0)
)

graphite_clean = graphite_clean[
    ["partner", "year", "import_value"]
]

print(graphite_clean.head())
print("Total import value:", graphite_clean["import_value"].sum())

                                           partner  year  import_value
1                             United Arab Emirates  2019       37362.0
27                                          Brazil  2019    17643422.0
34                                          Canada  2019     1656343.0
39  Switzerland (incl. Liechtenstein 'LI' -> 1994)  2019       32968.0
44                                           China  2019    24977156.0
Total import value: 1628501848.0


###Step 16 - Standardise Lithium Carbonate

In [ ]:
lithium_clean = lithium_all[
    ["partner", "year", "import_value"]
].copy()

print(lithium_clean.head())
print("Total import value:", lithium_clean["import_value"].sum())

                                          partner  year  import_value
0                                       Argentina  2019    17396516.0
1                                          Canada  2019        1550.0
2  Switzerland (incl. Liechtenstein 'LI' -> 1994)  2019        2003.0
3                                           Chile  2019   118643663.0
4                                           China  2019     1962367.0
Total import value: 2540510230.0


###Step 17 - Standardise Copper

In [ ]:
copper_clean = copper_all.copy()

copper_clean["import_value"] = (
    copper_clean["product_26030000"].fillna(0) +
    copper_clean["product_74010000"].fillna(0) +
    copper_clean["product_74020000"].fillna(0) +
    copper_clean["product_74031100"].fillna(0) +
    copper_clean["product_74031300"].fillna(0)
)

copper_clean = copper_clean[
    ["partner", "year", "import_value"]
]

print(copper_clean.head())
print("Total import value:", copper_clean["import_value"].sum())

                partner  year  import_value
0  United Arab Emirates  2019         552.0
1               Armenia  2019    72504741.0
2                Angola  2019     6061271.0
3             Australia  2019   183405168.0
4            Azerbaijan  2019      983194.0
Total import value: 202338448817.0


###Step 18 -  Combine the five materials

In [ ]:
magnesium_clean = magnesium_clean.groupby(
    ["partner", "year"], as_index=False
)["import_value"].sum()

gallium_clean = gallium_clean.groupby(
    ["partner", "year"], as_index=False
)["import_value"].sum()

graphite_clean = graphite_clean.groupby(
    ["partner", "year"], as_index=False
)["import_value"].sum()

lithium_clean = lithium_clean.groupby(
    ["partner", "year"], as_index=False
)["import_value"].sum()

copper_clean = copper_clean.groupby(
    ["partner", "year"], as_index=False
)["import_value"].sum()

magnesium_clean["material"] = "Magnesium"
gallium_clean["material"] = "Gallium"
graphite_clean["material"] = "Natural graphite"
lithium_clean["material"] = "Lithium carbonate"
copper_clean["material"] = "Copper"

all_materials = pd.concat(
    [
        magnesium_clean,
        gallium_clean,
        graphite_clean,
        lithium_clean,
        copper_clean
    ],
    ignore_index=True
)

all_materials = all_materials.rename(
    columns={
        "partner": "country",
        "import_value": "import_value_eur"
    }
)

all_materials = all_materials[
    ["material", "country", "year", "import_value_eur"]
]

print("Combined dataset shape:", all_materials.shape)
print(all_materials.head())

Combined dataset shape: (1064, 4)
    material     country  year  import_value_eur
0  Magnesium   Argentina  2022             121.0
1  Magnesium   Australia  2023            3474.0
2  Magnesium   Australia  2024            7133.0
3  Magnesium     Bahrain  2021           74394.0
4  Magnesium  Bangladesh  2019            1021.0


###Step 19 - Validate the combined dataset

In [ ]:
print("Missing values:")
print(all_materials.isna().sum())

print("\nDuplicate material-country-year records:",
      all_materials.duplicated(
          subset=["material", "country", "year"]
      ).sum())

print("\nYears available:")
print(sorted(all_materials["year"].unique()))

print("\nRecords and total import value by material:")
print(
    all_materials.groupby("material")["import_value_eur"]
    .agg(["count", "sum"])
)

print("\nNegative import values:",
      (all_materials["import_value_eur"] < 0).sum())

Missing values:
material            0
country             0
year                0
import_value_eur    0
dtype: int64

Duplicate material-country-year records: 0

Years available:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Records and total import value by material:
                   count           sum
material                              
Copper               485  2.023384e+11
Gallium               74  1.249070e+08
Lithium carbonate    126  2.540510e+09
Magnesium            132  5.778941e+09
Natural graphite     247  1.628502e+09

Negative import values: 0


In [ ]:
countries = sorted(all_materials["country"].unique())

print("Number of unique partner names:", len(countries))

for country in countries:
    print(country)

Number of unique partner names: 120
Albania
Algeria
Angola
Argentina
Armenia
Australia
Azerbaijan
Bahamas
Bahrain
Bangladesh
Belarus (Belorussia)
Bolivia, Plurinational State of
Bosnia and Herzegovina
Botswana
Brazil
Burkina Faso
Burundi
Cambodia
Cameroon
Canada
Central African Republic
Ceuta (incl. Melilla 'XL' -> 1998)
Chad
Chile
China
Colombia
Congo
Congo, Democratic Republic of
Costa Rica
Countries and territories not specified for commercial or military reasons in the framework of extra-Union trade
Countries and territories not specified within the framework of extra-Union trade
Countries and territories not specified within the framework of intra-Union trade
Curaçao
Dominican Republic
Ecuador
Egypt
El Salvador
Eritrea
Ethiopia (incl. Eritrea 'ER' -> 1993)
Extra-EU27 (from 2020)
French Polynesia
Gabon
Georgia
Ghana
Gibraltar
Greenland
Guatemala
High seas
Honduras
Hong Kong
Iceland
India
Indonesia (incl. East Timor 'TP' from 1977 -> 2000)
Iran, Islamic Republic of
Israel (incl. Wes

###Step 20 - Classify partner entries

In [ ]:
all_materials = all_materials.rename(
    columns={"country": "partner_area"}
)

unspecified_partners = [
    "Countries and territories not specified for commercial or military reasons in the framework of extra-Union trade",
    "Countries and territories not specified within the framework of extra-Union trade",
    "Countries and territories not specified within the framework of intra-Union trade"
]

all_materials["partner_type"] = "Named partner area"

all_materials.loc[
    all_materials["partner_area"] == "Extra-EU27 (from 2020)",
    "partner_type"
] = "EU aggregate"

all_materials.loc[
    all_materials["partner_area"].isin(unspecified_partners),
    "partner_type"
] = "Unspecified partner"

all_materials.loc[
    all_materials["partner_area"] == "High seas",
    "partner_type"
] = "Special category"

print(
    all_materials.groupby("partner_type")
    .agg(
        records=("partner_area", "count"),
        total_import_value_eur=("import_value_eur", "sum")
    )
)

                     records  total_import_value_eur
partner_type                                        
EU aggregate              35            1.062028e+11
Named partner area      1004            1.061918e+11
Special category           1            3.500000e+03
Unspecified partner       24            1.673556e+07


In [ ]:
partner_totals = (
    all_materials[all_materials["partner_type"] != "EU aggregate"]
    .groupby(["material", "year"])["import_value_eur"]
    .sum()
    .reset_index(name="partner_total")
)

eu_totals = (
    all_materials[all_materials["partner_type"] == "EU aggregate"]
    [["material", "year", "import_value_eur"]]
    .rename(columns={"import_value_eur": "eu_aggregate"})
)

reconciliation = eu_totals.merge(
    partner_totals,
    on=["material", "year"]
)

reconciliation["difference_pct"] = (
    (reconciliation["partner_total"] - reconciliation["eu_aggregate"])
    / reconciliation["eu_aggregate"] * 100
)

print(
    reconciliation.groupby("material")["difference_pct"]
    .agg(["min", "max"])
    .round(4)
)

                   min     max
material                      
Copper             0.0  0.0373
Gallium            0.0  0.0000
Lithium carbonate  0.0  0.0000
Magnesium          0.0  0.0002
Natural graphite   0.0  0.0000


In [ ]:
supplier_data = all_materials[
    all_materials["partner_type"] == "Named partner area"
].copy()

supplier_data = supplier_data[
    ["material", "partner_area", "year", "import_value_eur"]
]

print("Supplier analysis dataset:", supplier_data.shape)
print("Unique partner areas:", supplier_data["partner_area"].nunique())
print(supplier_data.groupby("material")["import_value_eur"].sum())

Supplier analysis dataset: (1004, 4)
Unique partner areas: 115
material
Copper               1.011574e+11
Gallium              6.245350e+07
Lithium carbonate    1.270255e+09
Magnesium            2.889469e+09
Natural graphite     8.122126e+08
Name: import_value_eur, dtype: float64


In [ ]:
for name, filename in files.items():
    print(name, ":", filename)

Magnesium : ds-045409__custom_22978252_spreadsheet.xlsx
Gallium : ds-045409__custom_22978287_spreadsheet.xlsx
Natural graphite : ds-045409__custom_22978295_spreadsheet.xlsx
Lithium carbonate : ds-045409__custom_22978315_spreadsheet.xlsx
Copper : ds-045409__custom_22978350_spreadsheet.xlsx


In [ ]:
import os

for filename in os.listdir():
    if filename.endswith((".xlsx", ".xls")):
        print(filename)

ds-045409__custom_22978287_spreadsheet.xlsx
ds-045409__custom_22978350_spreadsheet.xlsx
ds-045409__custom_22978315_spreadsheet.xlsx
ds-045409__custom_22978295_spreadsheet.xlsx
ds-045409__custom_22978252_spreadsheet.xlsx
CRM_update_June_2026_v2.xlsx


In [ ]:
crm_file = "CRM_update_June_2026_v2.xlsx"

crm_excel = pd.ExcelFile(crm_file)

print("Number of sheets:", len(crm_excel.sheet_names))

for sheet in crm_excel.sheet_names:
    print(sheet)

Number of sheets: 22
Figure 1
Figure 2
Table 1
Figure 3
Table 2
Figure 4
Table 3
Table 4
Figure 5
Figure 6
Table 5
Figure 7
Table 6
Figure 8
Table 7
Figure 9
Table 8
Figure 10
Table 9
Figure 11
Table 10
Table 11


In [ ]:
for sheet in crm_excel.sheet_names:
    if sheet.startswith("Table"):
        df = pd.read_excel(crm_file, sheet_name=sheet, header=None)

        print("\n---", sheet, "---")
        print("Shape:", df.shape)
        print(df.head(8).to_string(index=False, header=False))


--- Table 1 ---
Shape: (24, 12)
Table 1: Largest partners of imports of REE+ by product code in 2025 in value              NaN      NaN         NaN      NaN           NaN      NaN                          NaN       NaN               NaN NaN            NaN
                                                                          NaN Largest partners      NaN         NaN      NaN           NaN      NaN Share in total trade of REE+       NaN             Ratio NaN            NaN
                                                                 Product code            First      NaN      Second      NaN         Third      NaN                      Imports    Export Imports / exports NaN Top 3 combined
                                                                     28469060           Norway 0.504643       China 0.301197         India 0.108233                     0.299272  0.297682          1.007753 NaN       0.914073
                                                                     28

In [ ]:
for sheet in crm_excel.sheet_names:
    if sheet.startswith("Figure"):
        df = pd.read_excel(crm_file, sheet_name=sheet, header=None)

        print("\n---", sheet, "---")
        print("Shape:", df.shape)
        print(df.dropna(how="all").head(8).to_string(index=False, header=False))


--- Figure 1 ---
Shape: (27, 2)
NaN Critical raw materials and their supply risk
NaN                Source: Joint Research Centre

--- Figure 2 ---
Shape: (12, 14)
REE+               NaN         NaN         NaN NaN                  NaN        NaN       NaN NaN          NaN       NaN        NaN NaN       NaN
 NaN Value (€ million)         NaN         NaN NaN Weight (1000 tonnes)        NaN       NaN NaN Price per kg       NaN        NaN NaN       NaN
 NaN              Year     Imports     Exports NaN                 Year    Imports   Exports NaN         Year   Imports    Exports NaN       NaN
 NaN              2021  106.499827  138.208368 NaN                 2021  16.918438  7.413622 NaN         2021  6.294897  18.642489 NaN       NaN
 NaN              2022  145.728436  141.617616 NaN                 2022  18.382846  6.828787 NaN         2022  7.927414  20.738327 NaN       NaN
 NaN              2023  123.861054  102.287652 NaN                 2023  18.336055  5.555693 NaN         2023 

In [ ]:
price_sheets = {
    "Magnesium": "Figure 3",
    "Gallium": "Figure 6",
    "Natural graphite": "Figure 8",
    "Copper": "Figure 9",
    "Lithium carbonate": "Figure 10"
}

price_frames = []

for material, sheet_name in price_sheets.items():

    df = pd.read_excel(crm_file, sheet_name=sheet_name, header=None)

    year_columns = []

    for col in range(df.shape[1]):
        if str(df.iloc[2, col]).strip() == "Year":
            year_columns.append(col)

    price_year_col = year_columns[2]

    prices = df.iloc[3:, [price_year_col, price_year_col + 1]].copy()
    prices.columns = ["year", "import_price_eur_per_kg"]

    prices["year"] = pd.to_numeric(prices["year"], errors="coerce")
    prices["import_price_eur_per_kg"] = pd.to_numeric(
        prices["import_price_eur_per_kg"], errors="coerce"
    )

    prices = prices.dropna()

    prices["year"] = prices["year"].astype(int)
    prices["material"] = material

    price_frames.append(prices)

crm_prices = pd.concat(price_frames, ignore_index=True)

crm_prices = crm_prices[
    ["material", "year", "import_price_eur_per_kg"]
]

print(crm_prices.sort_values(
    ["material", "year"]
).to_string(index=False))

print("\nRecords per material:")
print(crm_prices.groupby("material").size())

         material  year  import_price_eur_per_kg
           Copper  2019                 2.392630
           Copper  2020                 2.273080
           Copper  2021                 3.141259
           Copper  2022                 3.423995
           Copper  2023                 3.180717
           Copper  2024                 3.565647
           Copper  2025                 3.910018
          Gallium  2019               174.015702
          Gallium  2020               173.730829
          Gallium  2021               175.881055
          Gallium  2022               305.133325
          Gallium  2023               332.274087
          Gallium  2024               356.516392
          Gallium  2025               363.247383
Lithium carbonate  2019                10.060168
Lithium carbonate  2020                 7.623961
Lithium carbonate  2021                 6.138800
Lithium carbonate  2022                20.761307
Lithium carbonate  2023                30.581333
Lithium carbonate  2

###Step 21 - Validate the price dataset

In [ ]:
print("Dataset shape:", crm_prices.shape)

print("\nMissing values:")
print(crm_prices.isna().sum())

print("\nDuplicate material-year records:",
      crm_prices.duplicated(["material", "year"]).sum())

print("\nPrice data by material:")
print(
    crm_prices.groupby("material").agg(
        years=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
)

print("\nNumeric data types:")
print(crm_prices.dtypes)

Dataset shape: (35, 3)

Missing values:
material                   0
year                       0
import_price_eur_per_kg    0
dtype: int64

Duplicate material-year records: 0

Price data by material:
                   years  first_year  last_year
material                                       
Copper                 7        2019       2025
Gallium                7        2019       2025
Lithium carbonate      7        2019       2025
Magnesium              7        2019       2025
Natural graphite       7        2019       2025

Numeric data types:
material                    object
year                         int64
import_price_eur_per_kg    float64
dtype: object


###Step 22 - Audit missing product values

In [ ]:
product_columns = {
    "Magnesium": ["product_81041100", "product_81041900"],
    "Natural graphite": ["product_250410", "product_250490"],
    "Copper": [
        "product_26030000",
        "product_74010000",
        "product_74020000",
        "product_74031100",
        "product_74031300"
    ]
}

product_datasets = {
    "Magnesium": magnesium_all,
    "Natural graphite": graphite_all,
    "Copper": copper_all
}

for material, df in product_datasets.items():

    columns = product_columns[material]
    missing_count = df[columns].isna().sum(axis=1)

    complete = (missing_count == 0).sum()
    partial = (
        (missing_count > 0) &
        (missing_count < len(columns))
    ).sum()

    print("\n", material)
    print("Total records:", len(df))
    print("All product values reported:", complete)
    print("Some product values missing:", partial)
    print("Missing values by product:")
    print(df[columns].isna().sum())


 Magnesium
Total records: 132
All product values reported: 68
Some product values missing: 64
Missing values by product:
product_81041100    33
product_81041900    31
dtype: int64

 Natural graphite
Total records: 247
All product values reported: 133
Some product values missing: 114
Missing values by product:
product_250410    40
product_250490    74
dtype: int64

 Copper
Total records: 485
All product values reported: 52
Some product values missing: 433
Missing values by product:
product_26030000     91
product_74010000    314
product_74020000    279
product_74031100    263
product_74031300    410
dtype: int64


In [ ]:
clean_datasets = {
    "Magnesium": magnesium_clean,
    "Natural graphite": graphite_clean,
    "Copper": copper_clean
}

for material, df in product_datasets.items():

    columns = product_columns[material]

    temp = df[["partner", "year"]].copy()
    temp["import_value"] = df[columns].sum(axis=1, min_count=1)

    recalculated = temp.groupby(
        ["partner", "year"], as_index=False
    )["import_value"].sum()

    previous = clean_datasets[material]

    comparison = recalculated.merge(
        previous,
        on=["partner", "year"],
        suffixes=("_new", "_previous")
    )

    difference = (
        comparison["import_value_new"] -
        comparison["import_value_previous"]
    ).abs().max()

    print(material, "- Maximum difference:", difference)

Magnesium - Maximum difference: 0.0
Natural graphite - Maximum difference: 0.0
Copper - Maximum difference: 0.0


In [ ]:
import zipfile

supplier_data.to_csv("supplier_data.csv", index=False)
crm_prices.to_csv("crm_prices.csv", index=False)
all_materials.to_csv("all_materials_reference.csv", index=False)
reconciliation.to_csv("reconciliation.csv", index=False)

export_files = [
    "supplier_data.csv",
    "crm_prices.csv",
    "all_materials_reference.csv",
    "reconciliation.csv"
]

with zipfile.ZipFile("phase1_data.zip", "w") as zip_file:
    for filename in export_files:
        zip_file.write(filename)

print("Phase 1 exports completed.")

print("Supplier data:", supplier_data.shape)
print("CRM prices:", crm_prices.shape)
print("Reference data:", all_materials.shape)
print("Reconciliation:", reconciliation.shape)

from google.colab import files as colab_files
colab_files.download("phase1_data.zip")

Phase 1 exports completed.
Supplier data: (1004, 4)
CRM prices: (35, 3)
Reference data: (1064, 5)
Reconciliation: (35, 5)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Phase 1 - Data Inspection, Extraction and Preparation

## 1. Purpose of Phase 1

The first phase focused on collecting, understanding and preparing reliable data for our EU Critical Raw Materials supply-chain analysis.

Before calculating supplier dependency or identifying supply-chain risks, we needed to make sure that the original data was extracted correctly, cleaned and organised into a consistent structure.

Our analysis covers five materials over seven years (2019–2025):

- Magnesium
- Gallium
- Natural graphite
- Lithium carbonate
- Copper

## 2. Data sources

We used two main sources.

**Source 1 — Eurostat Comext (DS-045409)**

We downloaded five material-specific Excel datasets containing annual EU import data from 2019 to 2025.

The data includes:
- Reporting partner countries and territories.
- Product codes.
- Annual import values in euros.

We extracted import data only. Export records were not included in our supplier-dependency dataset.

**Source 2 — CRM reference workbook**

We used `CRM_update_June_2026_v2.xlsx` to obtain annual import prices per kg for the five materials.

The relevant worksheets were:

| Material | Worksheet |
|---|---|
| Magnesium | Figure 3 |
| Gallium | Figure 6 |
| Natural graphite | Figure 8 |
| Copper | Figure 9 |
| Lithium carbonate | Figure 10 |

## 3. Data extraction and cleaning

We used Python and pandas in Google Colab to inspect the Excel files and extract the relevant information.

The main preparation activities included:

- Identifying the correct annual worksheets and product-code columns.
- Extracting import records for 2019–2025.
- Converting import values into numeric formats.
- Handling `:` entries as missing values rather than automatically treating them as zero.
- Removing records without usable supplier and import information.
- Checking missing values, duplicate records and available years.
- Combining multiple product codes belonging to the same material.

Some materials contain multiple product codes. We combined their reported import values to calculate the total imports from each partner for each year.

For example, magnesium contains two product codes, while copper contains five.

We retained the original product-level information so it can be revisited in more detailed analysis.

## 4. Building a combined supplier dataset

After preparing the five individual datasets, we combined them into one consistent dataset.

The final supplier-level structure contains:

| Column | Description |
|---|---|
| `material` | Critical raw material |
| `partner_area` | Reporting partner country or territory |
| `year` | Import year |
| `import_value_eur` | Annual import value in euros |

The combined dataset initially contained **1,064 records**.

We also identified different types of partner entries:

- Named partner countries and territories.
- The `Extra-EU27` aggregate.
- Unspecified partner categories.
- Special categories such as `High seas`.

The EU aggregate is not an individual supplier. We therefore retained it in the reference dataset for validation but excluded it from the supplier-level analysis.

The resulting supplier dataset contains:

- 1,004 records.
- 115 distinct named partner areas.
- Five materials.
- Seven years of import data.

We retained named territories and geographical partner areas rather than automatically excluding them because they are not sovereign states.

## 5. Data validation and reconciliation

We performed several checks to confirm that the prepared data was suitable for further analysis.

The combined dataset passed the following checks:

- No missing material names, partner names or years.
- No duplicate material–partner–year records.
- All seven years from 2019 to 2025 were present.
- No negative import values.
- All import-value columns were numeric.

We also compared the partner-level import totals against the `Extra-EU27` aggregate for each material and year.

The results showed very close agreement:

- Gallium: 0% maximum difference.
- Lithium carbonate: 0% maximum difference.
- Magnesium: 0.0002% maximum difference.
- Natural graphite: 0% maximum difference.
- Copper: 0.0373% maximum difference.

These results provide a strong cross-check of the extracted trade values. The small copper discrepancy is retained rather than artificially adjusted.

## 6. Preparation of import price data

We separately extracted annual import prices per kg from the CRM reference workbook.

The resulting dataset, `crm_prices`, contains:

| Column | Description |
|---|---|
| `material` | Critical raw material |
| `year` | Import year |
| `import_price_eur_per_kg` | Annual import price per kg in euros |

The dataset contains **35 observations**: five materials across seven years.

Validation confirmed:

- No missing values.
- No duplicate material–year records.
- Seven years for every material.
- Numeric price values.

These figures represent aggregate import unit values reported in the CRM workbook. They should not be interpreted as supplier-specific prices or market spot prices.

## 7. Important data limitations and decisions

A few points need to remain visible throughout the project.

**Missing product values**

Some supplier-year records have a reported value for one product code but a missing value for another.

| Material | All product values reported | Some product values missing |
|---|---:|---:|
| Magnesium | 68 | 64 |
| Natural graphite | 133 | 114 |
| Copper | 52 | 433 |

We did not remove these records simply because one product value was missing.

We also checked the effect of using reported values directly versus replacing missing values with zero during product aggregation. The maximum difference was zero for all three materials.

However, this calculation does not prove that every missing source value represents zero trade. The original missing-value limitation must be acknowledged.

**Price limitations**

The price dataset is available at material level, not at individual supplier level. Therefore, we can examine how import prices changed over time, but we cannot directly compare the price paid to each supplier using this dataset.

**Trade measurement**

Our supplier-dependency analysis is based on import values in euros. It measures value-based dependency, not physical-volume dependency.

---

# Phase 1 Outcome

Phase 1 has established the two datasets required for the next analytical stages:

1. `supplier_data` — prepared country/partner-level import data.
2. `crm_prices` — annual import prices per kg.

We also retained `all_materials` as a reference dataset, including aggregate and unidentified partner categories, and created `reconciliation` for validation.

The final CSV executed and downloaded.

---

# What Comes Next?

## Phase 2 - Supplier Dependency and Concentration Analysis

The objective is to understand how dependent the EU is on individual supplier countries for each critical raw material.

We will calculate:

- Total annual imports for each material.
- Individual supplier import shares (%).
- Largest supplier for each material.
- Top three supplier shares.
- Herfindahl-Hirschman Index (HHI).

The HHI will help us distinguish materials with diversified supplier bases from materials concentrated among a small number of suppliers.

## Phase 3 - Historical Trends and Risk Assessment

We will examine how supplier dependency changed between 2019 and 2025.

Questions we want to answer:

- Is dependence on China increasing or decreasing?
- Are alternative suppliers gaining importance?
- Which materials have become more concentrated?
- Which materials have experienced significant changes in import prices?
- Are there years where supplier concentration and price movements occurred together?

We will distinguish observed trends from explanations that require additional evidence.

## Phase 4 - Business Analysis and Diversification Priorities

This is where we turn the numerical results into meaningful supply-chain insights.

We will identify:

- Materials with the highest supplier concentration.
- Materials with increasing dependency.
- Materials exposed to limited alternative suppliers.
- Priority materials for supplier diversification.
- Potential supply-chain risks and practical mitigation strategies.

We will develop recommendations based on the actual findings rather than assuming that the material with the highest import value is automatically the riskiest.

## Phase 5-— Visualisation and Final Presentation

Finally, we will communicate the analysis through professional charts, dashboards and a structured business report.

The final output should answer one central question:

**Which EU critical raw materials present the greatest supplier-concentration risks, how have those risks changed from 2019 to 2025, and where should the EU prioritise supplier diversification?**

---

**Overall objective:** Build a credible, evidence-based supply-chain risk analysis using real Eurostat trade data, Python and business-oriented interpretation. The goal is not simply to produce charts or calculate HHI, but to understand what the numbers mean for European manufacturing and procurement decisions.